# G1版 早期片麻痺モデル — 半身の運動出力を段階的に弱め、転倒閾値を算出する

このノートブックは、**脳卒中発症初期の、片麻痺による歩行障害**を、G1のpolicyで再現する
ことを目的とします。

**設計方針(これまでの研究プログラムとの関係)**:
- **運動出力(出力層ニューロン)を、半身(13関節: 脚6+腕7)まとめて、同時に、
  段階的に弱める**(単一関節ではなく、"半身全体"という規模で、しかも運動出力軸で
  弱めるのは、本ノートブックが初めて)
- **policyは、一切、再学習させない。** これは、以前確認した通り、既存の関連文献
  (Choi et al. 2026等)が、すべて「弱めた"後"に、再学習・再最適化する」という、
  慢性期を模した設計だったのに対し、**「弱めたまま、そのまま歩かせる」ことで、
  発症"直後"(神経系がまだ代償戦略を獲得していない段階)を模す**、という、
  これまでの一貫した方針を踏襲します
- **「転倒するか、しないか」の、明確な閾値を算出**します(粗いスイープ→二分探索)

構成:
1. セットアップ
2. 関節名の確認・actorへのアクセス確認
3. 半身・段階的弱化の実装
4. 粗いスイープ(閾値のおおよその範囲を特定)
5. 二分探索(閾値を精密に特定)
6. 左右差の分析


## 1. セットアップ

In [ ]:
import os
os.environ["MUJOCO_GL"] = "egl"
os.environ["PYOPENGL_PLATFORM"] = ""
os.environ["WANDB_MODE"] = "offline"

%pip install -q "mjlab==1.2.0" "mujoco==3.5.0" "warp-lang==1.12.1" "numpy>=2.0,<2.3" scipy mediapy pandas

import torch
assert torch.cuda.is_available(), "GPU ランタイムに切り替えてください"
print("✅ install done:", torch.cuda.get_device_name(0))


## 2. 関節名の確認・actorへのアクセス確認

In [ ]:
import math, glob, re, os
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import asdict
from mjlab.envs import ManagerBasedRlEnv
from mjlab.rl import MjlabOnPolicyRunner, RslRlVecEnvWrapper
from mjlab.tasks.registry import load_env_cfg, load_rl_cfg

TASK = "Mjlab-Velocity-Flat-Unitree-G1"
DEVICE = "cuda:0"
ENV_SEED = 100
RIGHT_FACING_YAW_RAD = -math.pi / 2

_probe_cfg = load_env_cfg(TASK, play=True)
_probe_cfg.scene.num_envs = 1
_probe_env = ManagerBasedRlEnv(cfg=_probe_cfg, device=DEVICE, render_mode=None)
JOINT_NAMES = list(_probe_env.scene["robot"].joint_names)
assert len(JOINT_NAMES) == 29
print("JOINT_NAMES を確定しました(29関節)。")

LEFT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("left_")]
RIGHT_HALF_JOINTS = [n for n in JOINT_NAMES if n.startswith("right_")]
assert len(LEFT_HALF_JOINTS) == 13 and len(RIGHT_HALF_JOINTS) == 13
print(f"左半身(13関節): {LEFT_HALF_JOINTS}")
print(f"右半身(13関節): {RIGHT_HALF_JOINTS}")

_probe_env.close()
del _probe_env
torch.cuda.empty_cache()


In [ ]:
# actor.mlp[-1](出力層)への、正しいアクセス経路を、実際に確認する
# (以前、g1_single_neuron_ablation.ipynbで確認済みの経路: runner.alg.actor.mlp)
_run_dirs = sorted(glob.glob("logs/rsl_rl/g1_velocity/*_seed1"))
assert _run_dirs, "seed1のrunディレクトリが見つかりません。学習済みcheckpointが必要です。"
_ckpt = max(Path(_run_dirs[-1]).glob("model_*.pt"),
            key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

_play_cfg = load_env_cfg(TASK, play=True)
_play_cfg.scene.num_envs = 1
_play_cfg.seed = ENV_SEED
_agent_cfg = load_rl_cfg(TASK)

_raw_env = ManagerBasedRlEnv(cfg=_play_cfg, device=DEVICE, render_mode=None)
_env = RslRlVecEnvWrapper(_raw_env, clip_actions=_agent_cfg.clip_actions)
_runner = MjlabOnPolicyRunner(_env, asdict(_agent_cfg), device=DEVICE)
_runner.load(str(_ckpt), load_cfg={"actor": True}, strict=True, map_location=DEVICE)

_final_layer = _runner.alg.actor.mlp[-1]
print("最終層:", _final_layer)
print("weight の形状:", _final_layer.weight.shape)
assert _final_layer.weight.shape[0] == 29
print("\n✅ runner.alg.actor.mlp[-1] への、正しいアクセスを確認しました。")

_env.close()
del _runner, _env, _raw_env
torch.cuda.empty_cache()


## 3. 半身・段階的弱化の実装

関節トルクの上限を下げるのではなく、**policyの出力層(最終層)の重みとバイアスを、対象の関節について直接、段階的に弱めます**。
既存の単一関節ablation研究の用量反応実験を、"単一関節"から"複数関節、同時"に拡張したものです。


In [ ]:
def load_policy_with_graded_half_ablation(checkpoint_path, joint_idx_list, degradation_level,
                                             env_seed=ENV_SEED, yaw_rad=RIGHT_FACING_YAW_RAD):
    """半身(複数関節)の、出力ニューロンを、まとめて、degradation_level倍する。
    degradation_level=1.0で正常、0.0で完全ablation(片麻痺の最重度に相当)。"""
    play_cfg = load_env_cfg(TASK, play=True)
    play_cfg.scene.num_envs = 1
    play_cfg.seed = env_seed
    play_cfg.events["reset_base"].params["pose_range"]["yaw"] = (yaw_rad, yaw_rad)
    agent_cfg = load_rl_cfg(TASK)

    raw_env = ManagerBasedRlEnv(cfg=play_cfg, device=DEVICE, render_mode=None)
    env = RslRlVecEnvWrapper(raw_env, clip_actions=agent_cfg.clip_actions)

    runner = MjlabOnPolicyRunner(env, asdict(agent_cfg), device=DEVICE)
    runner.load(str(checkpoint_path), load_cfg={"actor": True}, strict=True, map_location=DEVICE)
    policy = runner.get_inference_policy(device=DEVICE)

    with torch.no_grad():
        final_layer = runner.alg.actor.mlp[-1]
        for idx in joint_idx_list:
            final_layer.weight[idx, :] *= degradation_level
            final_layer.bias[idx] *= degradation_level

    return runner, policy, env, raw_env


def run_hemiparesis_rollout(checkpoint_path, joint_idx_list, degradation_level,
                              forward_speed_mps=0.5, steps=250, env_seed=ENV_SEED):
    runner, policy, env, raw_env = load_policy_with_graded_half_ablation(
        checkpoint_path, joint_idx_list, degradation_level, env_seed=env_seed
    )
    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)

    obs = env.get_observations()
    fell_over = False
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)
        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    n_steps_completed = step + 1
    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()

    return {"fell_over": fell_over, "n_steps_completed": n_steps_completed}


## 4. 粗いスイープ(閾値のおおよその範囲を特定)

**まず、6段階(100%・80%・60%・40%・20%・0%)で、8seed、速度0.5m/s固定**で、
転倒率を確認します。resume可能な設計です。


In [ ]:
HEMI_CSV = "g1_hemiparesis_coarse.csv"
HEMI_LEVELS = [1.0, 0.8, 0.6, 0.4, 0.2, 0.0]
HEMI_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]
HEMI_SPEED = 0.5

HEMI_CONDITIONS = {
    "right_half_paresis": [JOINT_NAMES.index(n) for n in RIGHT_HALF_JOINTS],
    "left_half_paresis":  [JOINT_NAMES.index(n) for n in LEFT_HALF_JOINTS],
}

if os.path.exists(HEMI_CSV):
    _existing_hemi_df = pd.read_csv(HEMI_CSV)
    hemi_completed = set(zip(_existing_hemi_df["condition"], _existing_hemi_df["seed"], _existing_hemi_df["level"]))
    print(f"既存のログを検出: {len(_existing_hemi_df)}行、{len(hemi_completed)}組が完了済み")
else:
    hemi_completed = set()
    print("新規に開始します。")

total_hemi_combos = len(HEMI_CONDITIONS) * len(HEMI_SEEDS) * len(HEMI_LEVELS)
n_done = 0

for condition_label, joint_idx_list in HEMI_CONDITIONS.items():
    for seed in HEMI_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            print(f"seed{seed}: runディレクトリが見つかりません。スキップします。")
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for level in HEMI_LEVELS:
            if (condition_label, seed, level) in hemi_completed:
                continue
            n_done += 1
            print(f"=== [{len(hemi_completed)+n_done}/{total_hemi_combos}] "
                  f"{condition_label} / seed{seed} / level={level} ===")
            result = run_hemiparesis_rollout(ckpt, joint_idx_list, level, HEMI_SPEED)
            print(f"  {result}")

            row_df = pd.DataFrame([{
                "condition": condition_label, "seed": seed, "level": level, **result
            }])
            write_header = not os.path.exists(HEMI_CSV)
            row_df.to_csv(HEMI_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{HEMI_CSV} に保存しました。")


In [ ]:
hemi_df = pd.read_csv(HEMI_CSV)

hemi_pivot = (
    hemi_df.groupby(["condition", "level"])["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_seeds"})
    .reset_index()
)
hemi_pivot["fall_rate"] = hemi_pivot["n_falls"] / hemi_pivot["n_seeds"]

print("=== 粗いスイープ、残存率ごとの転倒率 ===")
hemi_table = hemi_pivot.pivot(index="level", columns="condition", values="fall_rate").sort_index(ascending=False)
print(hemi_table.to_string())

print()
print("次の§5(二分探索)で、より精密な閾値を特定する範囲を、上記の表を見て決めてください。")
print("(例: 60%では0%、40%では転倒あり、なら、40-60%の間で二分探索する)")


## 5. 二分探索(閾値を精密に特定)

**§4の結果を見て、転倒率が0%から上昇し始める、"残存率の範囲"を特定**してください。
以下のコードは、**その範囲の中で、二分探索を行い、"半数のseedが転倒する、境界の
残存率"を、より精密に絞り込みます。**

**`BISECTION_LOW`・`BISECTION_HIGH`を、§4の結果に応じて、書き換えてから実行して
ください。**


In [ ]:
# --- §4の結果を見て、ここを調整してください ---
BISECTION_LOW = 0.6   # 粗いスイープで転倒の移行を挟む区間の下端(改訂時に修正)
BISECTION_HIGH = 0.8  # 同・上端(改訂時に修正)
BISECTION_STEPS = 4   # 二分探索を繰り返す回数(4回で、範囲は1/16に絞り込まれる)
# ----------------------------------------------

BISECT_CSV = "g1_hemiparesis_bisection.csv"
bisect_rows = []

for condition_label, joint_idx_list in HEMI_CONDITIONS.items():
    low, high = BISECTION_LOW, BISECTION_HIGH
    for step_i in range(BISECTION_STEPS):
        mid = round((low + high) / 2, 4)
        print(f"\n=== {condition_label}: 二分探索 {step_i+1}/{BISECTION_STEPS}回目、残存率={mid} ===")

        n_falls_at_mid = 0
        for seed in HEMI_SEEDS:
            run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
            if not run_dirs:
                continue
            ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                       key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
            result = run_hemiparesis_rollout(ckpt, joint_idx_list, mid, HEMI_SPEED)
            n_falls_at_mid += int(result["fell_over"])
            bisect_rows.append({"condition": condition_label, "level": mid, "seed": seed,
                                  "bisection_round": step_i + 1, **result})

        fall_rate_at_mid = n_falls_at_mid / len(HEMI_SEEDS)
        print(f"  残存率={mid}: 転倒率={fall_rate_at_mid*100:.1f}%({n_falls_at_mid}/{len(HEMI_SEEDS)})")

        # 転倒率が50%以上なら、"もっと残す(highを上げる)"方向、50%未満なら"もっと弱める"方向へ絞り込む
        if fall_rate_at_mid >= 0.5:
            low = mid
        else:
            high = mid

    print(f"{condition_label}: 最終的な、転倒閾値の推定範囲 = [{low}, {high}]")

bisect_df = pd.DataFrame(bisect_rows)
bisect_df.to_csv(BISECT_CSV, index=False)
print(f"\n完了。{BISECT_CSV} に保存しました。")


## 6. 左右差の分析

In [ ]:
print("=== 二分探索の結果、各条件の、詳細な履歴 ===")
for condition_label in HEMI_CONDITIONS:
    sub = bisect_df[bisect_df["condition"] == condition_label]
    print(f"\n{condition_label}:")
    for round_i in sorted(sub["bisection_round"].unique()):
        round_sub = sub[sub["bisection_round"] == round_i]
        level = round_sub["level"].iloc[0]
        fall_rate = round_sub["fell_over"].mean()
        print(f"  ラウンド{round_i}: 残存率={level}, 転倒率={fall_rate*100:.1f}%")

print()
print("=== 左右の、転倒閾値の比較 ===")
print("上記の履歴から、右半身麻痺・左半身麻痺、それぞれの、最終的な閾値の推定範囲を比較し、")
print("どちらがより低い残存率(=より弱い麻痺)で転倒するか(=より脆弱か)を確認してください。")


## 7. 閾値付近の、精密な8seedスイープ+多数決による、頑健な左右差判定

二分探索(§5)で見つかった、おおよその閾値範囲(右: 約65-67.5%、左: 約61-62.5%)を
挟むように、**5段階(55%・60%・65%・70%・75%)を、8seed全部、速度0.5m/s固定**で
スイープします。これにより、**各seedごとに、"個別の閾値"を推定し、これまでの
研究プログラムと同じ、多数決による、頑健な左右差判定**を行います。resume可能な設計です。


In [ ]:
FINE_CSV = "g1_hemiparesis_fine.csv"
FINE_LEVELS = [0.55, 0.60, 0.65, 0.70, 0.75]
FINE_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]
FINE_SPEED = 0.5

if os.path.exists(FINE_CSV):
    _existing_fine_df = pd.read_csv(FINE_CSV)
    fine_completed = set(zip(_existing_fine_df["condition"], _existing_fine_df["seed"], _existing_fine_df["level"]))
    print(f"既存のログを検出: {len(_existing_fine_df)}行、{len(fine_completed)}組が完了済み")
else:
    fine_completed = set()
    print("新規に開始します。")

total_fine_combos = len(HEMI_CONDITIONS) * len(FINE_SEEDS) * len(FINE_LEVELS)
n_done = 0

for condition_label, joint_idx_list in HEMI_CONDITIONS.items():
    for seed in FINE_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for level in FINE_LEVELS:
            if (condition_label, seed, level) in fine_completed:
                continue
            n_done += 1
            print(f"=== [{len(fine_completed)+n_done}/{total_fine_combos}] "
                  f"{condition_label} / seed{seed} / level={level} ===")
            result = run_hemiparesis_rollout(ckpt, joint_idx_list, level, FINE_SPEED)
            print(f"  {result}")

            row_df = pd.DataFrame([{
                "condition": condition_label, "seed": seed, "level": level, **result
            }])
            write_header = not os.path.exists(FINE_CSV)
            row_df.to_csv(FINE_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{FINE_CSV} に保存しました。")


In [ ]:
fine_df = pd.read_csv(FINE_CSV)

fine_pivot = (
    fine_df.groupby(["condition", "level"])["fell_over"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "n_falls", "count": "n_seeds"})
    .reset_index()
)
fine_pivot["fall_rate"] = fine_pivot["n_falls"] / fine_pivot["n_seeds"]
print("=== 精密スイープ、残存率ごとの転倒率(8seed) ===")
print(fine_pivot.pivot(index="level", columns="condition", values="fall_rate").sort_index(ascending=False).to_string())

# 各seedごとに、「転倒しなかった、最も低い残存率」を、個別の閾値の目安とする
print()
print("=== seedごとの、個別の閾値(転倒しなかった、最も低い残存率) ===")
seed_thresholds = {}
for condition_label in HEMI_CONDITIONS:
    seed_thresholds[condition_label] = {}
    for seed in FINE_SEEDS:
        sub = fine_df[(fine_df["condition"] == condition_label) & (fine_df["seed"] == seed)]
        survived = sub[~sub["fell_over"]]
        thr = survived["level"].min() if len(survived) > 0 else float("inf")
        seed_thresholds[condition_label][seed] = thr
        print(f"  {condition_label} / seed{seed}: 個別閾値 ≈ {thr}")

print()
print("=== 多数決による、左右差の判定(個別閾値が高い方=より脆弱) ===")
right_more_fragile = left_more_fragile = tied = 0
for seed in FINE_SEEDS:
    r_thr = seed_thresholds["right_half_paresis"][seed]
    l_thr = seed_thresholds["left_half_paresis"][seed]
    if r_thr > l_thr:
        right_more_fragile += 1
    elif l_thr > r_thr:
        left_more_fragile += 1
    else:
        tied += 1
print(f"右がより脆弱(閾値が高い)={right_more_fragile}/8, "
      f"左がより脆弱(閾値が高い)={left_more_fragile}/8, 同等={tied}/8")


## 8. 速度依存性: 転倒閾値は、速度によって、どう動くか

これまでの閾値の推定は、すべて0.5 m/s固定でした。ここでは、**複数の速度(0.3・0.5・0.7・0.9 m/s)で、同じ精密スイープ(55-75%)を行い、閾値が速度に応じてどう変化するかを確認**します。

最初に4 seedで探索的に実行し、その後8 seed全部に拡張しました。resumeの仕組みにより、既に完了しているseed分は自動的にスキップされます。


In [ ]:
SPEED_CSV = "g1_hemiparesis_speed_dependence.csv"
SPEED_LEVELS = [0.55, 0.60, 0.65, 0.70, 0.75]
SPEED_LIST = [0.3, 0.5, 0.7, 0.9]
SPEED_SEEDS = [1, 2, 3, 4, 5, 6, 7, 8]  # 4seed→8seedへ拡張(resumeにより、1-4は自動的にスキップされる)

if os.path.exists(SPEED_CSV):
    _existing_sp_df = pd.read_csv(SPEED_CSV)
    sp_completed = set(zip(_existing_sp_df["condition"], _existing_sp_df["seed"],
                             _existing_sp_df["level"], _existing_sp_df["speed_mps"]))
    print(f"既存のログを検出: {len(_existing_sp_df)}行、{len(sp_completed)}組が完了済み")
else:
    sp_completed = set()
    print("新規に開始します。")

total_sp_combos = len(HEMI_CONDITIONS) * len(SPEED_SEEDS) * len(SPEED_LEVELS) * len(SPEED_LIST)
n_done = 0

for condition_label, joint_idx_list in HEMI_CONDITIONS.items():
    for seed in SPEED_SEEDS:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))

        for speed in SPEED_LIST:
            for level in SPEED_LEVELS:
                if (condition_label, seed, level, speed) in sp_completed:
                    continue
                n_done += 1
                print(f"=== [{len(sp_completed)+n_done}/{total_sp_combos}] "
                      f"{condition_label} / seed{seed} / level={level} / {speed}m/s ===")
                result = run_hemiparesis_rollout(ckpt, joint_idx_list, level, speed)
                print(f"  {result}")

                row_df = pd.DataFrame([{
                    "condition": condition_label, "seed": seed, "level": level,
                    "speed_mps": speed, **result
                }])
                write_header = not os.path.exists(SPEED_CSV)
                row_df.to_csv(SPEED_CSV, mode="a", header=write_header, index=False)

print(f"\n完了。{SPEED_CSV} に保存しました。")


In [ ]:
speed_df = pd.read_csv(SPEED_CSV)

print("=== 速度ごとの、閾値付近の転倒率(4seed) ===")
for condition_label in HEMI_CONDITIONS:
    print(f"\n--- {condition_label} ---")
    sub = speed_df[speed_df["condition"] == condition_label]
    table = sub.groupby(["speed_mps", "level"])["fell_over"].mean().reset_index()
    table = table.pivot(index="level", columns="speed_mps", values="fell_over").sort_index(ascending=False)
    print(table.to_string())

print()
print("=== 速度ごとの、推定閾値(転倒率が50%を超える、最も高い残存率) ===")
for condition_label in HEMI_CONDITIONS:
    print(f"\n{condition_label}:")
    sub = speed_df[speed_df["condition"] == condition_label]
    for speed in SPEED_LIST:
        speed_sub = sub[sub["speed_mps"] == speed]
        table = speed_sub.groupby("level")["fell_over"].mean().sort_index(ascending=False)
        over_half = table[table >= 0.5]
        est_threshold = over_half.index.max() if len(over_half) > 0 else "55%未満"
        print(f"  {speed}m/s: 推定閾値 ≈ {est_threshold}")


## 9. 機序解明: 半身麻痺での転倒も、"デフォルト姿勢への崩壊"か

これまでの、単一関節・感覚喪失の研究と、同じ手法(**転倒直前の、全29関節の角度と、
`default_joint_pos`の比較**)を、半身麻痺条件に適用します。**麻痺した半身だけでなく、
健常な半身までもが、デフォルトに収束するか**が、特に重要な問いです。


> **改訂時の注記(Revision note)**: この§9の計測は、`env.step()` の**後**に関節角を読んでいるため、転倒した試行では
> 環境の自動リセット後(=デフォルト姿勢)の値を記録してしまいます。原稿の初版で報告した「デフォルト姿勢への収束」は、
> このためのアーティファクトでした。正しい計測(リセット前の状態を記録)は、同梱の `g1_hemiparesis_revision.ipynb` の
> §B・§Cを参照してください。この節は、経緯を示すために残しています。
>
> *This section reads joint angles after `env.step()` returns and therefore records the post-reset (default) posture on the
> termination step. The corrected measurement is in `g1_hemiparesis_revision.ipynb`, Sections B and C.*

In [ ]:
def run_hemiparesis_rollout_with_full_log(checkpoint_path, joint_idx_list, degradation_level,
                                              forward_speed_mps=0.5, steps=250, env_seed=ENV_SEED):
    runner, policy, env, raw_env = load_policy_with_graded_half_ablation(
        checkpoint_path, joint_idx_list, degradation_level, env_seed=env_seed
    )
    forward_cmd = torch.tensor([[forward_speed_mps, 0.0, 0.0]], device=DEVICE)
    obs = env.get_observations()
    rows = []
    fell_over = False
    for step in range(steps):
        raw_env.command_manager.get_term("twist").vel_command_b[:] = forward_cmd
        with torch.no_grad():
            action = policy(obs)
        obs, _, dones, _ = env.step(action)

        joint_pos = raw_env.scene["robot"].data.joint_pos[0].detach().cpu().numpy()
        row = {"step": step}
        for i, name in enumerate(JOINT_NAMES):
            row[f"jpos_{name}_deg"] = np.degrees(joint_pos[i])
        rows.append(row)

        fell = bool(dones[0].item()) if hasattr(dones[0], "item") else bool(dones[0])
        if fell:
            fell_over = True
            break

    env.close()
    del runner, policy, env, raw_env
    torch.cuda.empty_cache()
    return pd.DataFrame(rows), fell_over


# 環境のデフォルト角度を取得する
_probe_cfg_m = load_env_cfg(TASK, play=True)
_probe_cfg_m.scene.num_envs = 1
_probe_env_m = ManagerBasedRlEnv(cfg=_probe_cfg_m, device=DEVICE, render_mode=None)
default_joint_pos_all = np.degrees(_probe_env_m.scene["robot"].data.default_joint_pos[0].cpu().numpy())
_probe_env_m.close()
del _probe_env_m
torch.cuda.empty_cache()

# 閾値付近(右=65%、左=60%)で、4seed、機序解析用の時系列を記録する
MECH_LEVEL_BY_CONDITION = {"right_half_paresis": 0.65, "left_half_paresis": 0.60}
mechanism_hemi_logs = {}

for condition_label, joint_idx_list in HEMI_CONDITIONS.items():
    level = MECH_LEVEL_BY_CONDITION[condition_label]
    for seed in [1, 2, 3, 4]:
        run_dirs = sorted(glob.glob(f"logs/rsl_rl/g1_velocity/*_seed{seed}"))
        if not run_dirs:
            continue
        ckpt = max(Path(run_dirs[-1]).glob("model_*.pt"),
                   key=lambda p: int(re.search(r"model_(\d+)", p.name).group(1)))
        print(f"=== {condition_label}(level={level}) / seed{seed} の時系列を記録中 ===")
        df, fell = run_hemiparesis_rollout_with_full_log(ckpt, joint_idx_list, level)
        mechanism_hemi_logs[(condition_label, seed)] = (df, fell)
        print(f"  {len(df)}ステップ、転倒={fell}")


In [ ]:
print("=== 転倒した試行について、全29関節の、終端角度とデフォルト角度の差(度)を確認 ===")
print()

for (condition_label, seed), (df, fell) in mechanism_hemi_logs.items():
    if not fell:
        print(f"{condition_label} / seed{seed}: 転倒せず、対象外\n")
        continue

    paretic_set = set(RIGHT_HALF_JOINTS if "right_half" in condition_label else LEFT_HALF_JOINTS)
    terminal = df.iloc[-1]

    diffs = []
    for i, name in enumerate(JOINT_NAMES):
        val = terminal[f"jpos_{name}_deg"]
        diff = val - default_joint_pos_all[i]
        diffs.append((name, name in paretic_set, diff))

    diffs_df = pd.DataFrame(diffs, columns=["joint", "is_paretic_side", "diff_from_default_deg"])
    diffs_df["abs_diff"] = diffs_df["diff_from_default_deg"].abs()

    paretic_mean_abs = diffs_df[diffs_df["is_paretic_side"]]["abs_diff"].mean()
    healthy_mean_abs = diffs_df[~diffs_df["is_paretic_side"]]["abs_diff"].mean()

    print(f"{condition_label} / seed{seed}(n_steps={len(df)}):")
    print(f"  麻痺側({len(diffs_df[diffs_df['is_paretic_side']])}関節)の、デフォルトとの平均絶対差: {paretic_mean_abs:.1f}度")
    print(f"  健常側({len(diffs_df[~diffs_df['is_paretic_side']])}関節)の、デフォルトとの平均絶対差: {healthy_mean_abs:.1f}度")
    print()
